In [1]:
import pandas as pd

In [2]:
party_code_df = pd.read_csv("partycode.csv")
party_code_df.head()

,account_id,bioguide_id,mem_name,screen_name,chamber,state_abbr,party_code
0,37007274,Y000033,Don Young,repdonyoung,House,AK,200
1,2559398984,Y000033,Don Young,DonYoungAK,House,AK,200
2,2253968388,B001289,Bradley Byrne,RepByrne,House,AL,200
3,42481696,B001289,Bradley Byrne,BradleyByrne,House,AL,200
4,2861616083,P000609,Gary Palmer,USRepGaryPalmer,House,AL,200


In [3]:
party_dict = party_code_df[['party_code','account_id']].set_index('account_id').to_dict()
party_dict = party_dict['party_code']

In [4]:
example_tweets = pd.read_json("mediabias_main/congresstweets/data/2020-04-27.json")
example_tweets.head()

,id,screen_name,user_id,time,link,text,source
0,1254621841353777152,JoaquinCastrotx,231510077,2020-04-27T00:02:11-04:00,https://www.twitter.com/RojelioDavila/statuses...,RT @RojelioDavila @KasieDC @JoaquinCastrotx I ...,Twitter Web App
1,1254636622701899776,HaleyLive,391445643,2020-04-27T01:00:55-04:00,https://www.twitter.com/HaleyLive/statuses/125...,@charles_gaba Sorry to hear this my friend.,Twitter for iPhone
2,1254634339209326592,JaredHuffman,334894942,2020-04-27T00:51:51-04:00,https://www.twitter.com/JaredHuffman/statuses/...,Not buying this one! https://twitter.com/jdick...,Twitter for Android
3,1254634026247192576,JaredHuffman,334894942,2020-04-27T00:50:36-04:00,https://www.twitter.com/RepAdamSchiff/statuses...,RT @RepAdamSchiff Was there ever a man born to...,Twitter for Android
4,1254632461625102336,CongressmanRaja,814179031956488192,2020-04-27T00:44:23-04:00,https://www.twitter.com/CongressmanRaja/status...,It’s not every day you hear an argument for xe...,Twitter for iPhone


In [5]:
example_tweets["party_code"] = example_tweets["user_id"]
example_tweets = example_tweets.replace({"party_code":party_dict})
example_tweets[['text','party_code']]
train_data = example_tweets[(example_tweets['party_code']!=100)|(example_tweets['party_code']!=200)][['text','party_code']]

In [6]:
# Rearrange rows
date_samples = 100
train_data.sample(frac=1)
train_data_R=train_data[train_data['party_code']==200].head(date_samples)
train_data_D=train_data[train_data['party_code']==100].head(date_samples)
def fix_party_code(pc):
    return int(pc/100-1)
train_data=train_data_D.append(train_data_R)
train_data['party_code'] = train_data['party_code'].apply(fix_party_code)
train_data

,text,party_code
0,RT @RojelioDavila @KasieDC @JoaquinCastrotx I ...,0
1,@charles_gaba Sorry to hear this my friend.,0
2,Not buying this one! https://twitter.com/jdick...,0
3,RT @RepAdamSchiff Was there ever a man born to...,0
4,It’s not every day you hear an argument for xe...,0
...,...,...
308,"By the way, @chiproytx asked me to add him as ...",1
322,.@RepAdamSchiff hates transparency. He won’t a...,1
325,RT @SBAgov TODAY: SBA will resume accepting #P...,1
328,🚨Economic Impact Payment (EIP) Alert 🚨 \n ...,1


In [7]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
nltk.download('punkt')

[nltk_data] Downloading package stopwords to C:\Users\Herman D
[nltk_data]     Schaumburg\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Herman D
[nltk_data]     Schaumburg\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [8]:
train_data['text'] = train_data['text'].apply(word_tokenize)

In [9]:
train_data

,text,party_code
0,"[RT, @, RojelioDavila, @, KasieDC, @, JoaquinC...",0
1,"[@, charles_gaba, Sorry, to, hear, this, my, f...",0
2,"[Not, buying, this, one, !, https, :, //twitte...",0
3,"[RT, @, RepAdamSchiff, Was, there, ever, a, ma...",0
4,"[It, ’, s, not, every, day, you, hear, an, arg...",0
...,...,...
308,"[By, the, way, ,, @, chiproytx, asked, me, to,...",1
322,"[., @, RepAdamSchiff, hates, transparency, ., ...",1
325,"[RT, @, SBAgov, TODAY, :, SBA, will, resume, a...",1
328,"[🚨Economic, Impact, Payment, (, EIP, ), Alert,...",1


# Text Vectorization

In [10]:
from os import system
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from joblib import dump, load # used for saving and loading sklearn objects
from scipy.sparse import save_npz, load_npz # used for saving and loading sparse matrices

In [11]:
system("mkdir data_preprocessors")
system("mkdir vectorized_data")

# Unigram Counts
def list_tostring(input_list):
    return ' '.join(input_list)
train_data['text'] = train_data['text'].apply(list_tostring)
unigram_vectorizer = CountVectorizer(ngram_range=(1, 1))
unigram_vectorizer.fit(train_data['text'].values)

CountVectorizer(analyzer='word', binary=False, decode_error='strict',
                dtype=<class 'numpy.int64'>, encoding='utf-8', input='content',
                lowercase=True, max_df=1.0, max_features=None, min_df=1,
                ngram_range=(1, 1), preprocessor=None, stop_words=None,
                strip_accents=None, token_pattern='(?u)\\b\\w\\w+\\b',
                tokenizer=None, vocabulary=None)

In [12]:
dump(unigram_vectorizer, 'data_preprocessors/unigram_vectorizer.joblib')

['data_preprocessors/unigram_vectorizer.joblib']

In [14]:
# unigram_vectorizer = load('data_preprocessors/unigram_vectorizer.joblib')

X_train_unigram = unigram_vectorizer.transform(train_data['text'].values)
save_npz('vectorized_data/X_train_unigram.npz', X_train_unigram)

In [15]:
# Unigram Tf-Idf

unigram_tf_idf_transformer = TfidfTransformer()
unigram_tf_idf_transformer.fit(X_train_unigram)

dump(unigram_tf_idf_transformer, 'data_preprocessors/unigram_tf_idf_transformer.joblib')

['data_preprocessors/unigram_tf_idf_transformer.joblib']

In [16]:
X_train_unigram_tf_idf = unigram_tf_idf_transformer.transform(X_train_unigram)

save_npz('vectorized_data/X_train_unigram_tf_idf.npz', X_train_unigram_tf_idf)

In [18]:
# Bigram Counts

bigram_vectorizer = CountVectorizer(ngram_range=(1, 2))
bigram_vectorizer.fit(train_data['text'].values)

dump(bigram_vectorizer, 'data_preprocessors/bigram_vectorizer.joblib')

['data_preprocessors/bigram_vectorizer.joblib']

In [20]:
X_train_bigram = bigram_vectorizer.transform(train_data['text'].values)

save_npz('vectorized_data/X_train_bigram.npz', X_train_bigram)

In [21]:
bigram_tf_idf_transformer = TfidfTransformer()
bigram_tf_idf_transformer.fit(X_train_bigram)

dump(bigram_tf_idf_transformer, 'data_preprocessors/bigram_tf_idf_transformer.joblib')

['data_preprocessors/bigram_tf_idf_transformer.joblib']

In [22]:
X_train_bigram_tf_idf = bigram_tf_idf_transformer.transform(X_train_bigram)

save_npz('vectorized_data/X_train_bigram_tf_idf.npz', X_train_bigram_tf_idf)

# SGDClassifier

In [24]:
def train_and_show_scores(X: csr_matrix, y: np.array, title: str) -> None:
    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, train_size=0.75, stratify=y
    )

    clf = SGDClassifier()
    clf.fit(X_train, y_train)
    train_score = clf.score(X_train, y_train)
    valid_score = clf.score(X_valid, y_valid)
    print(f'{title}\nTrain score: {round(train_score, 2)} ; Validation score: {round(valid_score, 2)}\n')

y_train = train_data['party_code'].values

In [25]:
train_and_show_scores(X_train_unigram, y_train, 'Unigram Counts')
train_and_show_scores(X_train_unigram_tf_idf, y_train, 'Unigram Tf-Idf')
train_and_show_scores(X_train_bigram, y_train, 'Bigram Counts')
train_and_show_scores(X_train_bigram_tf_idf, y_train, 'Bigram Tf-Idf')

Unigram Counts
Train score: 1.0 ; Validation score: 0.72

Unigram Tf-Idf
Train score: 1.0 ; Validation score: 0.74

Bigram Counts
Train score: 1.0 ; Validation score: 0.76

Bigram Tf-Idf
Train score: 1.0 ; Validation score: 0.64



# Using Cross-Validation for hyperparameter tuning

In [30]:

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

X_train = X_train_bigram


# Phase 1: loss, learning rate and initial learning rate

clf = SGDClassifier()

distributions = dict(
    loss=['hinge', 'log', 'modified_huber', 'squared_hinge', 'perceptron'],
    learning_rate=['optimal', 'invscaling', 'adaptive'],
    eta0=uniform(loc=1e-7, scale=1e-2)
)

random_search_cv = RandomizedSearchCV(
    estimator=clf,
    param_distributions=distributions,
    cv=5,
    n_iter=50
)
random_search_cv.fit(X_train, y_train)
print(f'Best params: {random_search_cv.best_params_}')
print(f'Best score: {random_search_cv.best_score_}')

Best params: {'eta0': 0.007375745421344656, 'learning_rate': 'invscaling', 'loss': 'log'}
Best score: 0.69


In [31]:
# Phase 2: penalty and alpha

clf = SGDClassifier()

distributions = dict(
    penalty=['l1', 'l2', 'elasticnet'],
    alpha=uniform(loc=1e-6, scale=1e-4)
)

random_search_cv = RandomizedSearchCV(
    estimator=clf,
    param_distributions=distributions,
    cv=5,
    n_iter=50
)
random_search_cv.fit(X_train, y_train)
print(f'Best params: {random_search_cv.best_params_}')
print(f'Best score: {random_search_cv.best_score_}')

Best params: {'alpha': 1.6428159575447975e-05, 'penalty': 'l1'}
Best score: 0.6599999999999999


## Saving best classifier

In [35]:
system("mkdir classifiers")

sgd_classifier = random_search_cv.best_estimator_

dump(random_search_cv.best_estimator_, 'classifiers/sgd_classifier.joblib')

# sgd_classifier = load('classifiers/sgd_classifier.joblib')

['classifiers/sgd_classifier.joblib']

# Testing Classifier

In [37]:
X_test = bigram_vectorizer.transform(train_data['text'].values)
X_test = bigram_tf_idf_transformer.transform(X_test)
y_test = train_data['party_code'].values

score = sgd_classifier.score(X_test, y_test)
print(score)

1.0
